In [1]:
import sys
 
# adding Folder_2 to the system path code/python/tfcat_tools/tfcat_tools
# sys.path.insert(0, 'tfcat_tools/')
sys.path.insert(0, '/obs/kkesri/code/python/tfcat_tools/')
sys.path.insert(0, '/obs/kkesri/code/ephem/')

In [2]:
import numpy 
import numpy.polynomial.polynomial as nppol
import glob
from tqdm import tqdm
from itertools import repeat
from datetime import datetime
from h5py import File
import multiprocessing
import os 
from scipy.io import readsav
from astropy.time import Time

from spacepy import pycdf
from matplotlib import pyplot as plt
import matplotlib.colors as colors
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from mpl_toolkits import axes_grid1
import matplotlib.gridspec as gridspec

from scipy.interpolate import interp1d
from tfcat import TFCat
from tfcat.validate import JSONSCHEMA_URI

#from tfcat_tools.tfcat_tools.edit import merge
#from tfcat_tools.tfcat_tools.mask import Mask
from tfcat_tools.edit import merge
from tfcat_tools.mask import Mask
from tfcat_tools import FeatureCollection

from read_juno_ephemeris_from_amda import *
from jupiter_ephem import *

import pandas as pd
from scipy.stats import skewnorm


In [3]:
def data_to_hdf5(file_name: str,
                time: numpy.ndarray,
                freqs: numpy.ndarray,
                data: numpy.ndarray,):
        """
        Saves the data to disk as a pre-processed HDF5 file.
        """
        output_file = File(file_name, 'w')
        output_file.attrs.create('observer', 'Juno')

        # Has to be done differently as this is an Astropy quantity
        output_file.create_dataset('Time', data=time)
        output_file['Time'].attrs.create('units', 'unix')

        output_file.create_dataset('Frequency', data=freqs)
        output_file['Frequency'].attrs.create('units',  'kHz')

        output_file.create_dataset('Data', data=data)
        output_file['Data'].attrs.create('units',  'W/m2/Hz')


        output_file.close()

def load_hdf5_data(file_name: str):
    """
    Reads a datafile in HDF5 format using the config file set earlier.
    """
    file: File = File(file_name)
    time = file['Time']
    freqs = file['Frequency']
    flux = file['Flux']
    return time, freqs, flux



def nozeros(x):
    return numpy.where(x != 0, x, 1)

def xyz_to_rtp(xyz):
    rtp = numpy.zeros(shape=xyz.shape)
    rtp[..., 0] = numpy.sqrt(xyz[..., 0]**2+xyz[..., 1]**2+xyz[..., 2]**2)
    rtp[..., 1] = numpy.arcsin(xyz[..., 2]/nozeros(rtp[..., 0]))*180/numpy.pi
    rtp[..., 2] = numpy.arctan2(xyz[..., 1], xyz[..., 0])*180/numpy.pi
    return rtp

def set_ephemeris(time, ephemeris, local_time_ephemeris):
    tmin, tmax = time.min(), time.max()
    new_ephemeris_time = ephemeris[0][(ephemeris[0] >= tmin) & (ephemeris[0] <= tmax)]
    new_local_time_ephemeris_time = local_time_ephemeris[0][(local_time_ephemeris[0] >= tmin) & (local_time_ephemeris[0] <= tmax)]
    _ephemeris_xyz = numpy.zeros(shape=(time.shape[0],3))
    _local_time_ephemeris_xyz = numpy.zeros(shape=(time.shape[0],3))
    for i in range(3) :
        _ephemeris = ephemeris[i+1][(ephemeris[0] >= tmin) & (ephemeris[0] <= tmax)]
        _ephemeris_xyz[:, i] = numpy.interp(time, new_ephemeris_time, _ephemeris)
        
        _ephemeris = local_time_ephemeris[i+1][(local_time_ephemeris[0] >= tmin) & (local_time_ephemeris[0] <= tmax)]
        _local_time_ephemeris_xyz[:, i]  = numpy.interp(time, new_local_time_ephemeris_time, _ephemeris)
        
    _ephemeris = numpy.zeros(shape=(time.shape[0],4))
    _ephemeris[:, 0:3] = xyz_to_rtp(_ephemeris_xyz)
    del _ephemeris_xyz
    _local_time_ephemeris = xyz_to_rtp(_local_time_ephemeris_xyz)
    _ephemeris[:, 3] = _local_time_ephemeris[:, 2]
    return _ephemeris

In [4]:
@numpy.vectorize
def webgeocalc_date_to_datetime(date_YMD,date_hours):
    return datetime.datetime.strptime(date_YMD+"T"+date_hours,"%Y-%m-%dT%H:%M:%S.%f")

@numpy.vectorize
def unix_date_to_datetime(date_unix):
    return datetime.datetime.fromtimestamp(date_unix)


def juno_ephemeris_from_webgeocalc(file, planetary_radius=False):
    if planetary_radius == False:
        print("### User needs to give a planetary radius to have ephemeris in planetary units ###")
    data_file = numpy.loadtxt(file,dtype="str")
    date = webgeocalc_date_to_datetime(data_file[:,0],data_file[:,1])
    x_coord = data_file[:,5].astype(float)
    y_coord = data_file[:,6].astype(float)
    z_coord = data_file[:,7].astype(float)

    return (date,x_coord[:]/planetary_radius,y_coord[:]/planetary_radius,z_coord[:]/planetary_radius)


In [5]:
# Function to read and concatenate variables from HDF5 files
def concatenate_hdf5_files(file_list, variable_names, path_files = './'):
    concatenated_data = {var: [] for var in variable_names}

    for file in file_list:
        with File(path_files+file, 'r') as f:
            for var in variable_names:
                concatenated_data[var].append(f[var][:])

    # Concatenate lists into numpy arrays
    concatenated_data = {var:numpy.concatenate(concatenated_data[var], axis=0) for var in variable_names}

    return concatenated_data


In [6]:
# Function to recursively flatten the nested structure
def flatten(nested_list):
    flat_list = []
    for item in nested_list:
        if isinstance(item, list):
            flat_list.extend(flatten(item))
        else:
            flat_list.append(item)
    return flat_list

In [7]:
def sort_cat(cat: TFCat):
    crs = cat.crs
    properties = cat.properties
    fields= cat.data.fields
    schema = cat.data["$schema"]
    event = FeatureCollection(cat.data, properties=properties, fields=fields, crs=crs, schema=schema).as_table()
    event.sort(['tmin'])
    return TFCat(FeatureCollection().from_table(table=event, properties=properties, fields=fields, crs=crs, schema=schema))

In [8]:
def load_intensity_from_hdf5_file(file_name: str):
    """
    Reads a datafile in HDF5 format using the config file set earlier.
    """
    file: File = File(file_name)
    normalized_intensity_bKOM = numpy.array(file['normalized_intensity_bKOM'])

    return normalized_intensity_bKOM       

In [9]:
from spacepy import pycdf
def load_background_data(cdf_file_path):
    with pycdf.CDF(cdf_file_path) as cdf_file:
        background = numpy.array(cdf_file["Background"])
        sigma = numpy.array(cdf_file["Sigma"])
        frequency = numpy.array(cdf_file["Frequency"])
        
        #for attr in cdf_file["Background"].attrs:        
        #    print(f"{attr}: {cdf_file['Background'].attrs[attr]}")
    
    return background, sigma, frequency


In [10]:
#thresholding data
def thresholding_data(data,background):

    occurrence = numpy.zeros(data.shape)
    for ifreq in range(len(background)):
        mask = data[:, ifreq] > background[ifreq]
        occurrence[mask, ifreq] = 1
    return occurrence

In [11]:
@numpy.vectorize
def from_UTC_to_astropy_Time(time_UTC):
    return Time(time_UTC[:-4]).unix

In [12]:
from webgeocalc import Calculation

def get_ephem_from_wgc(observer, time, api = None):
    """
    Get ephemeris data from WGC servers through API
    :param observer:
    :type observer: dict
    :param time:
    :type time: dict
    :return:
    :rtype: dict
    """

    wgc_kernels = [
                    "pds/wgc/mk/ground_stations_v0018.tm",
                    "pds/wgc/mk/solar_system_v0056.tm",
                    "pds/wgc/mk/latest_lsk_v0004.tm",
                    ]

    time_interval = [time['time_start'].strftime("%Y-%m-%d %H:%M:%S"), time['time_end'].strftime("%Y-%m-%d %H:%M:%S")]                    

    if api == None:
        api = ''
    elif api.lower() == 'obspm':
        api = 'https://voparis-webgeocalc2.obspm.fr/geocalc/api'
        #api = "https://wgc.jpl.nasa.gov:8443/webgeocalc/api"
        #api = 'https://wgc.obspm.fr/webgeocalc/api'
    calc = Calculation(
        api = api,
        kernel_paths =  wgc_kernels,
        intervals = time_interval,
        time_step = time['time_step'],
        time_step_units = time['time_units'],
        calculation_type = 'STATE_VECTOR',
        target = observer['NAME'],
        observer = observer['PARENT'],
        reference_frame = observer['referenceFrame'],
        aberration_correction = 'NONE',
        state_representation = 'PLANETOGRAPHIC',
    )

    calc.payload
    calc.run()

    longitude = numpy.array(calc.results['LONGITUDE'])  # must be in LongW for ExPRES
    latitude = numpy.array(calc.results['LATITUDE'])
    distance = numpy.array(calc.results['ALTITUDE'])
    time_ephem = numpy.array(calc.results['DATE'])

    result_data = {
                    'time': time_ephem,
                    'longitude': -longitude, # So that the longitude is counting West (SIII for Jupiter)
                    'latitude': latitude,
                    'distance': distance,

                }
    return result_data
    

Path to data, catalogue and ephemeris files

In [13]:
#PATHS
year = [2016,2017,2018,2019,2020,2021]
#year = [2020,2021]
path_cat = {f'{iyear}': f'/obs/kkesri/code/data/TFCat_catalog_2.1/juno_waves_lira_tfcat_{iyear}.json' for iyear in year}
path_dir_data  = '/obs/juno_data_preprocessed/'
path_dir_ephem = '/obs/kkesri/code/ephem/'
data_path_IAU_ephemeris = f"{path_dir_ephem}juno_jup_xyz_iau_2016_2025.txt"
data_path_JSO_ephemeris = f"{path_dir_ephem}juno_jup_xyz_jso_2016_2025.txt"

Definition of output path and keywords

In [14]:
directory_path_out = 'outputs/'
savefig = True

Constants Definition

In [15]:
AU = 149597870.700 # 1 astronomical unit in km
RJ = 71492. # Jovian radius in km

Loading Ephemeris

In [16]:
all_ephemeris = juno_ephemeris_from_amda(data_path_IAU_ephemeris)
local_time_ephemeris = juno_ephemeris_from_amda(data_path_JSO_ephemeris)

#print(threshold)

Time range definition

In [17]:
time_start = datetime.datetime(2016,6,1)
time_end = datetime.datetime(2021, 3, 1, 0, 0, 0) # or After datetime.datetime(2021, 9, 1, 0, 0, 0). Gap in between

HDF5 data files reading

In [18]:
hdf5_files_2016 = ["jno_wav_cdr_lesia_2016_apr_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2016_sep_dec.preprocessed.hdf5"]
hdf5_files_2017 = ["jno_wav_cdr_lesia_2017_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2017_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2017_sep_dec.preprocessed.hdf5"]
hdf5_files_2018 = ["jno_wav_cdr_lesia_2018_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2018_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2018_sep_dec.preprocessed.hdf5"]
hdf5_files_2019 = ["jno_wav_cdr_lesia_2019_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2019_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2019_sep_dec.preprocessed.hdf5"]
hdf5_files_2020 = ["jno_wav_cdr_lesia_2020_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2020_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2020_sep_dec.preprocessed.hdf5"]
hdf5_files_2021 = ["jno_wav_cdr_lesia_2021_jan_apr.preprocessed.hdf5",
"jno_wav_cdr_lesia_2021_may_aug.preprocessed.hdf5",
"jno_wav_cdr_lesia_2021_sep_dec.preprocessed.hdf5"]

# /!\ be careful, do not take into account beg. of 2021. bKOM is inverse mask, so if catalogue not done, mask takes everything
# /!\ do not take data at PJ, so wh en d < 2 R_Jupiter

# Dictionary to hold file lists for each year
files_dict = {
    2016: hdf5_files_2016,
    2017: hdf5_files_2017,
    2018: hdf5_files_2018,
    2019: hdf5_files_2019,
    2020: hdf5_files_2020,
    2021: hdf5_files_2021,
}


# Variables to concatenate
variables = ['Flux', 'Time']

# Dictionary to store concatenated data for each year
yearly_data = {}

# Process each year
for iyear, files in files_dict.items():
    yearly_data[iyear] = concatenate_hdf5_files(files, variables, path_files = path_dir_data)


Frequency reading

In [19]:
with File(path_dir_data+hdf5_files_2016[0]) as hdf_file:
    Frequency = numpy.array(hdf_file['Frequency'])

Data variables and TFCat catalogues sorting and creation of dictionnaries

In [20]:
file_name_hdf5 = {f'{iyear}': f'mask_{iyear}.hdf5' for iyear in year}
cat = {f'{iyear}': sort_cat(TFCat.from_file(path_cat[f'{iyear}'])) for iyear in year}
for iyear in year:
    cat[f'{iyear}'].properties['instrument_name'] = "Waves_" 

time = {f'{iyear}': numpy.array(Time(yearly_data[iyear]['Time'], format="jd").unix) for iyear in year}
data = {f'{iyear}': numpy.array(yearly_data[iyear]['Flux']) for iyear in year}

Selection in time

In [21]:
mask_time_start = time[time_start.strftime("%Y")] >= Time(time_start,format="datetime").unix
time[time_start.strftime("%Y")] = time[time_start.strftime("%Y")][mask_time_start]
data[time_start.strftime("%Y")] = data[time_start.strftime("%Y")][mask_time_start,:]

mask_time_end = time[time_end.strftime("%Y")] <= Time(time_end,format="datetime").unix
time[time_end.strftime("%Y")] = time[time_end.strftime("%Y")][mask_time_end]
data[time_end.strftime("%Y")] = data[time_end.strftime("%Y")][mask_time_end,:]

Ephemerides calculation only on the wanted time range

In [22]:
time_arrays = list(time.values())
time_over_year_sec = numpy.concatenate(time_arrays)
# Normalization at 1 AU
mask_time = (all_ephemeris[0]>= time_start) & (all_ephemeris[0]< time_end)
ephem_x = all_ephemeris[1][mask_time]
ephem_y = all_ephemeris[2][mask_time]
ephem_z = all_ephemeris[3][mask_time]
ephem_date = all_ephemeris[0][mask_time]

distance = numpy.sqrt(ephem_x**2 + ephem_y**2 + ephem_z**2)
lat = (numpy.pi-numpy.arccos(ephem_z / distance)) * 180/numpy.pi
lon = numpy.pi-numpy.arctan2(ephem_y,ephem_x) * 180/numpy.pi

distance = numpy.sqrt(ephem_x**2 + ephem_y**2 + ephem_z**2)
lat = 90 - (numpy.arccos(ephem_z / distance))*180/numpy.pi 
lon = (360 - ((numpy.arctan2(ephem_y,ephem_x))*180/numpy.pi)) % 360

time_ephem_sec = Time(ephem_date, format='datetime').unix
oblon=numpy.interp(time_over_year_sec,time_ephem_sec,lon)
oblat=numpy.interp(time_over_year_sec,time_ephem_sec,lat) 
obdist=numpy.interp(time_over_year_sec,time_ephem_sec,distance)
obmlat=10.31*numpy.cos((196.61-oblon)*numpy.pi/180.)+oblat

In [23]:
print(Time(time_over_year_sec[0], format = "unix").isot, Time(time_over_year_sec[-1], format = "unix").isot,
        Time(time_ephem_sec[0], format = "unix").isot, Time(time_ephem_sec[-1], format = "unix").isot)

2016-06-01T00:00:08.184 2021-02-28T23:59:09.184 2016-06-01T00:05:00.000 2021-02-28T23:55:00.000


Frequency range selection

In [24]:
#time_index = numpy.where((time.datetime >= time_start) & (time.datetime < time_end))[0]

index_freq_LFR = numpy.where((Frequency > 1) & (Frequency < 142))[0][0:-1]
index_freq_bKOM = numpy.where((Frequency > 10) & (Frequency < 142))[0][0:-1]
index_freq_DAM = numpy.where(Frequency >3000)[0]

Calculating or loading normalized_intensity_bKOM

In [25]:
calculate_intensity = False
if calculate_intensity:
    intensity_bKOM_list = []

    for index, iyear in enumerate(year):
        print(f"{iyear}")
        mask_juno_bKOM = Mask(
                            cat[f'{iyear}'],
                            time[f'{iyear}'],
                            Frequency[index_freq_bKOM],
                            data[f'{iyear}'][:,index_freq_bKOM]
                            )

        mask_juno_bKOM_data =  mask_juno_bKOM.apply('QP','nLF','nKOM','QP?','nLF?','nKOM?','anom',invert=True)
        mask_juno_bKOM_data = mask_juno_bKOM_data * 377
        #integrated_intensity_bKOM = numpy.sum(mask_juno_bKOM_data, axis=1)

        intensity_bKOM_list.append(mask_juno_bKOM_data)

        #if index == 0:
        #    integrated_intensity_bKOM_over_year = integrated_intensity_bKOM
        #else:
        #    integrated_intensity_bKOM_over_year = [integrated_intensity_bKOM_over_year, integrated_intensity_bKOM]
    intensity_bKOM_over_year = numpy.concatenate(intensity_bKOM_list)

    #integrated_intensity_bKOM_over_year_flattened_arrays = flatten(integrated_intensity_bKOM_over_year)
    #normalized_integrated_intensity_bKOM = numpy.concatenate(integrated_intensity_bKOM_over_year_flattened_arrays)*(obdist/AU)**2
    normalized_intensity_bKOM = intensity_bKOM_over_year * (obdist[:, numpy.newaxis]*RJ / AU)**2    

    output_file = File('normalized_intensity_bKOM.hdf5', 'w')
    output_file.attrs.create('observer', 'Juno')
    output_file.create_dataset('normalized_intensity_bKOM', data=normalized_intensity_bKOM)
    output_file['normalized_intensity_bKOM'].attrs.create('units', 'W/m2/Hz@1AU')
    output_file.close()
else:
    normalized_intensity_bKOM = load_intensity_from_hdf5_file("/obs/kkesri/code/data/normalized_intensity_bKOM_catalog_v20.hdf5")


Filtering with distance

In [26]:
mask_distance = obdist < 10
normalized_intensity_bKOM[mask_distance,:] = numpy.nan

Calculating background from normalized_intensity_bKOM data

In [27]:
background_from_data = numpy.zeros(normalized_intensity_bKOM.shape[1])
for ifreq in range(normalized_intensity_bKOM.shape[1]):
    mask = normalized_intensity_bKOM[:,ifreq] > 0
    background_from_data[ifreq] =  numpy.nanmean(normalized_intensity_bKOM[mask, ifreq])
    

Calculating integrated intensity over frequency

In [28]:
normalized_integrated_intensity_bKOM = numpy.sum(normalized_intensity_bKOM, axis=1)

Calculating integrated occurrence over frequency using background data

In [29]:
occurrence = thresholding_data(normalized_intensity_bKOM, background_from_data)
integrated_occurrence = numpy.sum(occurrence, axis = 1)
mask = integrated_occurrence > 0
integrated_occurrence[mask] = 1

In [31]:
time_datetime = pd.to_datetime(time_over_year_sec, unit='s')

Resampling the Juno data to change the resolution from 1 min to 1 hour (roughly)

case = 0: When you want all the arrays sliced according to the timestamps of your choosing

case = 1: When you just the entire resampled/original data

In [ ]:
def resample_hourly(start, end, case): 
    df_juno = pd.DataFrame({ 
        "time": pd.to_datetime(time_over_year_sec, unit='s'), 
        "intensity": normalized_integrated_intensity_bKOM 
        }) 
    
    df_juno.set_index("time", inplace=True) 
    df_hourly = df_juno.resample("1H").mean() 
    
    df_dist = pd.DataFrame({ 
        "time": pd.to_datetime(time_over_year_sec, unit='s'), 
        "dist": obdist 
        }) 
    
    df_dist.set_index("time", inplace=True) 
    df_dist_hourly = df_dist.resample("1H").mean() 
    df_combined = df_hourly.join(df_dist_hourly) 
    df_subset = df_combined.loc[start:end] 
    
    time_subset = df_subset.index 
    intensity_subset = df_subset["intensity"] 
    dist_subset = df_subset["dist"] 
    
    if case == 0:  # Gives back only the sliced data
        return time_subset, intensity_subset, dist_subset
    
    if case == 1:  # Gives back the entire dataset
        return df_hourly
    

def sample_minute(start, end, case):
    if case == 0:  # Gives back the sliced data
        mask = (time_datetime >= start) & (time_datetime <= end)

        time_subset = time_datetime[mask]
        intensity_subset = normalized_integrated_intensity_bKOM[mask]
        dist_subset = numpy.array(obdist)[mask]
        
        return time_subset, intensity_subset, dist_subset
    
    if case == 1:  # Gives back the entire dataset
        time = time_datetime
        intensity = normalized_integrated_intensity_bKOM
        dist = numpy.array(obdist)

        return time, intensity, dist

Choosing to mark only a few equally spaced date, time and position coordinates in the entire interval 

In [ ]:
def ticks(time_subset, dist_subset):
    n_ticks = 8
    indices = numpy.linspace(0, len(time_subset)-1, n_ticks, dtype=int)

    tick_times = time_subset[indices]
    tick_labels = [
        f"{t.strftime('%Y-%m-%d')}\n{d:.2f} Rj"
        for t, d in zip(tick_times, dist_subset[indices])
    ]
    return tick_times, tick_labels

Reading Matt's File

In [ ]:
def readMMESH_fromFile(fullfilename):
    """
    Parameters
    ----------
    fullfilename : string
        The fully qualified (i.e., absolute) filename pointing to the MMESH 
        output .csv file

    Returns
    -------
    mmesh_results : pandas DataFrame
        The results of the MMESH run
    """
    
    #   Read the csv, parsing MultiIndex columns correctly
    mmesh_results = pd.read_csv(fullfilename, 
                                comment='#', header=[0,1], index_col=[0])
    
    #   Reset the index to be datetimes, rather than integers
    mmesh_results.index = pd.to_datetime(mmesh_results.index)
    
    return mmesh_results

output = readMMESH_fromFile("/obs/kkesri/code/data/MMESH_data/MMESH_JupiterMME.csv")
df=pd.DataFrame(output)

In [ ]:
def MMESH_slicing(start, end, diag: str):
    diag_available = ['B_mag','B_mag_a', 'B_mag_loc', 'B_mag_scale', 
                      'n_tot','n_tot_a', 'n_tot_loc', 'n_tot_scale',
                      'P_dyn','P_dyn_a', 'P_dyn_loc', 'P_dyn_scale',
                      'u_mag','u_mag_a', 'u_mag_loc', 'u_mag_scale']
    
    if diag not in diag_available:
        return df.loc[start:end]
        #raise ValueError(f"{diag} not found")
    
    i = diag_available.index(diag)
    return df.loc[start:end, df.columns[i]]

Function defined to higlight a certain stripped area in the timeseries which has to be either excluded or included. 

If you want to show excluded or deleted data: choose case = 0

If you want to show special or inclusive data: choose case = 1

In [ ]:
def highlighted_regions(parameter, quantity, case, time):
    if case == 0:
        mask_excluded = quantity < parameter
    if case == 1:
        mask_excluded = quantity > parameter
    
    excluded_regions = []
    in_region = False
    region_start = None  

    for i in range(len(mask_excluded)):
        if mask_excluded[i] and not in_region:
            region_start = time[i]  
            in_region = True
        elif not mask_excluded[i] and in_region:
            region_end = time[i]    
            excluded_regions.append((region_start, region_end))
            in_region = False

    if in_region:
        excluded_regions.append((region_start, time[-1]))

    return excluded_regions